# 25 — NMF → WJ Simplex 512

Nonnegative matrix factorization baseline. Unlike PCA, NMF naturally produces nonnegative latent factors, so the output can be L1-normalized and compared with Weighted Jaccard.

This is a strong classical baseline for nonnegative dense vectors.


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np

sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup,
    eval_recall,
    l1_simplex,
    load_dataset,
    nmslib_neighbors,
    recall_at_k,
    rerank_wj_gpu,
    save_result,
    shifted_l1_simplex,
)

# Edit here
dataset_name = "full"       # "10k" or "full"
out_dim = 512
THREADS = 150
seed = 42
run_rerank = True
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]
np.random.seed(seed)

METHOD_NAME = "nmf_wj_512"
NOTEBOOK_NAME = "25_nmf_wj_512.ipynb"
OUT_PATH = "/tmp/results_sota_nmf_wj_512.pkl"


In [ ]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
max_k = max(max(candidate_ks), 500)


In [ ]:
try:
    from sklearn.decomposition import MiniBatchNMF
    NMFClass = MiniBatchNMF
    kwargs = {"batch_size": 512, "max_iter": 200, "init": "random", "random_state": seed, "verbose": 1}
except ImportError:
    from sklearn.decomposition import NMF
    NMFClass = NMF
    kwargs = {"max_iter": 200, "init": "random", "random_state": seed, "verbose": 1}

fit_rows = min(len(corpus_qt), 50000)
print(f"Fitting {NMFClass.__name__} on {fit_rows:,} corpus rows -> {out_dim} dims")
nmf = NMFClass(n_components=out_dim, **kwargs)
nmf.fit(np.maximum(corpus_qt[:fit_rows], 0))

def transform_chunks(x, batch_size=512):
    chunks = []
    for start in range(0, len(x), batch_size):
        chunks.append(nmf.transform(np.maximum(x[start:start + batch_size], 0)).astype(np.float32))
    return np.vstack(chunks)

z = transform_chunks(qt)
embs = l1_simplex(z)


In [ ]:
corpus_embs = embs[:query_start]
query_embs = embs[query_start:]
print(f"embs={embs.shape} | simplex sums: {embs.sum(axis=1).min():.4f} .. {embs.sum(axis=1).max():.4f}")
print(f"corpus vector memory = {corpus_embs.nbytes / 1024**2:.1f} MB")

nbrs, ann_info = nmslib_neighbors(
    corpus_embs,
    query_embs,
    space="WeightedJaccard",
    k=max_k,
    threads=THREADS,
)
metrics = {
    **eval_recall(gt, nbrs, query_start, max_k),
    **ann_info,
    "dim": int(corpus_embs.shape[1]),
    "vec_mb": float(corpus_embs.nbytes / 1024**2),
}
print("\nNo rerank")
for k, v in metrics.items():
    if isinstance(k, int):
        print(f"R@{k:<4} = {v:.4f}")
print(f"QPS={metrics['qps']:.1f}")
results = {METHOD_NAME: metrics}

if run_rerank:
    preload_rerank_corpus(corpus_qt, corpus_sums)
    for ck in candidate_ks:
        print(f"\nRaw-WJ rerank from top-{ck}")
        cand, cand_info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
        t0 = time.time()
        rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
        qps_total = len(query_qt) / max(time.time() - t0 + len(query_qt) / max(cand_info['qps'], 1e-9), 1e-9)
        rr_metrics = {
            **eval_recall(gt, rr, query_start, ck),
            "qps": qps_total,
            "qps_candidates": cand_info["qps"],
            "candidate_k": ck,
        }
        for k, v in rr_metrics.items():
            if isinstance(k, int):
                print(f"R@{k:<4} = {v:.4f}")
        print(f"QPS={qps_total:.1f}")
        results[f"{METHOD_NAME}_rerank_{ck}"] = rr_metrics

release_rerank_corpus()
for key, value in results.items():
    save_result(OUT_PATH, dataset_name, key, value, meta={"notebook": NOTEBOOK_NAME})
cleanup()
